# SpeechSR V2 — Colab Training Notebook

Full pipeline for **ProposedModelV2** including:
- Optional fastMRI pretraining
- Generator pretraining with perceptual loss + patch-based data
- GAN fine-tuning with PatchGAN discriminator + k-space consistency
- Temporal inference for dynamic speech volumes
- Quick evaluation (PSNR / SSIM) with side-by-side visualisation

**Before running:** Enable a GPU runtime — Runtime → Change runtime type → T4 GPU (free tier).

See `UPDATES.md` in the repo for full architecture details.

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/SpeechSR'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/marioknicola/SpeechSR.git {REPO_DIR}
    %cd {REPO_DIR}

!pip install -r requirements.txt -q

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1. Configuration

Edit the paths and flags below before running any training cells.

In [ ]:
# ── Data paths (on your Google Drive) ────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive'
INPUT_DIR    = f'{DRIVE_ROOT}/SpeechSR_data/Synth_LR'   # 128×128 LR NIfTI files
TARGET_DIR   = f'{DRIVE_ROOT}/SpeechSR_data/HR'         # 512×512 HR NIfTI files
OUTPUT_DIR   = f'{DRIVE_ROOT}/SpeechSR_outputs'         # all checkpoints go here

# ── Subject split ─────────────────────────────────────────────────────────────
SUBJECTS     = '0021 0022 0023 0024 0025 0026 0027'
VAL_SUBJECT  = '0022'
TEST_SUBJECT = '0021'

# ── Training settings ─────────────────────────────────────────────────────────
PATCH_SIZE       = 64        # LR patch size (→ 512 HR patch at 8× scale)
BATCH_SIZE       = 2         # reduce to 1 if OOM on T4
NUM_WORKERS      = 2
PROPOSED_SIZE    = 1024      # HR target size (k-space zero-padded)

# ── Stage 1: Generator pretraining ────────────────────────────────────────────
PRETRAIN_EPOCHS  = 100
PRETRAIN_LR      = 1e-4

# ── Stage 2: GAN fine-tuning ──────────────────────────────────────────────────
GAN_EPOCHS       = 200
GAN_LR           = 1e-4
LAMBDA_ADV       = 0.01      # start small; increase to 0.05 if still blurry
LAMBDA_KSPACE    = 0.1
ALPHA_PERCEP     = 0.1       # perceptual loss weight

# ── Existing checkpoint to warm-start from (leave empty to train from scratch) ─
GENERATOR_CHECKPOINT = ''    # e.g. f'{OUTPUT_DIR}/pretrained/best_model.pth'

print('Configuration set.')
print(f'  Input:  {INPUT_DIR}')
print(f'  Target: {TARGET_DIR}')
print(f'  Output: {OUTPUT_DIR}')

---
## 2. (Optional) fastMRI Pretraining

Download the fastMRI brain single-coil dataset from https://fastmri.org/dataset/ and upload it to your Drive.  
This step is optional but strongly recommended when you have only 5 training subjects.

Skip this section if you already have a pretrained checkpoint.

In [ ]:
FASTMRI_DIR          = f'{DRIVE_ROOT}/fastmri/brain_singlecoil_train'  # .h5 files
FASTMRI_OUTPUT_DIR   = f'{OUTPUT_DIR}/fastmri_pretrained'
FASTMRI_EPOCHS       = 50
FASTMRI_BATCH_SIZE   = 4

In [ ]:
!python pretrain_fastmri.py \
    --data-dir        "{FASTMRI_DIR}" \
    --output-dir      "{FASTMRI_OUTPUT_DIR}" \
    --epochs          {FASTMRI_EPOCHS} \
    --batch-size      {FASTMRI_BATCH_SIZE} \
    --patch-size      {PATCH_SIZE} \
    --augment \
    --alpha-percep    {ALPHA_PERCEP} \
    --num-workers     {NUM_WORKERS}

# Use this checkpoint to warm-start the speech MRI training
GENERATOR_CHECKPOINT = f'{FASTMRI_OUTPUT_DIR}/best_model.pth'
print(f'fastMRI pretrain done. Checkpoint: {GENERATOR_CHECKPOINT}')

---
## 3. GAN Training

Two-stage training on speech MRI subjects:
1. **Stage 1** — generator pretraining with `ForegroundEdgePerceptualLoss` (no discriminator)
2. **Stage 2** — adversarial fine-tuning with PatchGAN + k-space consistency

If you want to run them separately (e.g. inspect Stage 1 results first), set `GAN_EPOCHS = 0` for Stage 1 only, then re-run with `PRETRAIN_EPOCHS = 0` and the Stage 1 checkpoint.

In [ ]:
GAN_OUTPUT_DIR = f'{OUTPUT_DIR}/gan'

ckpt_flag = f'--generator-checkpoint "{GENERATOR_CHECKPOINT}"' if GENERATOR_CHECKPOINT else ''

!python train_gan.py \
    --input-dir           "{INPUT_DIR}" \
    --target-dir          "{TARGET_DIR}" \
    --output-dir          "{GAN_OUTPUT_DIR}" \
    --subjects            {SUBJECTS} \
    --val-subject         {VAL_SUBJECT} \
    --test-subject        {TEST_SUBJECT} \
    --proposed-target-size {PROPOSED_SIZE} \
    --patch-size          {PATCH_SIZE} \
    --augment \
    --pretrain-epochs     {PRETRAIN_EPOCHS} \
    --pretrain-lr         {PRETRAIN_LR} \
    --gan-epochs          {GAN_EPOCHS} \
    --gan-lr              {GAN_LR} \
    --lambda-adv          {LAMBDA_ADV} \
    --lambda-kspace       {LAMBDA_KSPACE} \
    --alpha-percep        {ALPHA_PERCEP} \
    --batch-size          {BATCH_SIZE} \
    --num-workers         {NUM_WORKERS} \
    {ckpt_flag}

---
## 3b. (Alternative) Standard Training — no GAN

Trains ProposedModelV2 with `ForegroundEdgePerceptualLoss` only.  
Faster and more stable than GAN training, but will not recover fine texture as well.

In [ ]:
STANDARD_OUTPUT_DIR = f'{OUTPUT_DIR}/proposed2_standard'
STANDARD_EPOCHS     = 500

!python train.py \
    --model               proposed2 \
    --input-dir           "{INPUT_DIR}" \
    --target-dir          "{TARGET_DIR}" \
    --output-dir          "{STANDARD_OUTPUT_DIR}" \
    --subjects            {SUBJECTS} \
    --val-subject         {VAL_SUBJECT} \
    --test-subject        {TEST_SUBJECT} \
    --proposed-target-size {PROPOSED_SIZE} \
    --patch-size          {PATCH_SIZE} \
    --augment \
    --perceptual-loss \
    --epochs              {STANDARD_EPOCHS} \
    --batch-size          {BATCH_SIZE} \
    --num-workers         {NUM_WORKERS}

---
## 4. Inference

Run the best checkpoint on your test subject. Use `--temporal-window 3` for dynamic volumes to enforce frame-to-frame consistency.

In [ ]:
# Point to whichever checkpoint you want to use
INFER_CHECKPOINT = f'{GAN_OUTPUT_DIR}/best_gan_generator.pth'
# INFER_CHECKPOINT = f'{STANDARD_OUTPUT_DIR}/proposed2_run/best_model.pth'

INFER_INPUT      = f'{DRIVE_ROOT}/SpeechSR_data/Synth_LR/LR_kspace_Subject0021_oh.nii'
INFER_OUTPUT_DIR = f'{OUTPUT_DIR}/inference'
INFER_OUTPUT     = f'{INFER_OUTPUT_DIR}/proposed2_Subject0021_oh.nii'

USE_TEMPORAL     = True   # set False for static phoneme images

In [ ]:
import os
os.makedirs(INFER_OUTPUT_DIR, exist_ok=True)

temporal_flag = '--temporal-window 3' if USE_TEMPORAL else '--temporal-window 1'

!python infer.py \
    --model      proposed2 \
    --checkpoint "{INFER_CHECKPOINT}" \
    --input      "{INFER_INPUT}" \
    --output     "{INFER_OUTPUT}" \
    {temporal_flag}

print(f'Output saved to: {INFER_OUTPUT}')

---
## 5. Quick Evaluation

Computes PSNR and SSIM on the inference output vs the HR ground truth, then displays an input / SR / HR side-by-side comparison.

In [ ]:
EVAL_SR_PATH  = INFER_OUTPUT
EVAL_HR_PATH  = f'{DRIVE_ROOT}/SpeechSR_data/HR/HR_kspace_Subject0021_oh.nii'
EVAL_LR_PATH  = INFER_INPUT
EVAL_FRAME    = 0   # which frame (or slice) to display

In [ ]:
import nibabel as nib
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

def load_frame(path, frame=0):
    data = nib.load(path).get_fdata(dtype=np.float32)
    if data.ndim == 2:
        data = data[:, :, np.newaxis]
    img = data[:, :, min(frame, data.shape[2] - 1)]
    img = img - img.min()
    if img.max() > 0:
        img /= img.max()
    return img

sr = load_frame(EVAL_SR_PATH, EVAL_FRAME)
hr = load_frame(EVAL_HR_PATH, EVAL_FRAME)

# Resize SR to HR size for comparison if needed
if sr.shape != hr.shape:
    import torch, torch.nn.functional as F
    sr_t = torch.from_numpy(sr).unsqueeze(0).unsqueeze(0)
    sr = F.interpolate(sr_t, size=hr.shape, mode='bilinear', align_corners=False).squeeze().numpy()

psnr_val = psnr(hr, sr, data_range=1.0)
ssim_val = ssim(hr, sr, data_range=1.0)
print(f'PSNR: {psnr_val:.2f} dB')
print(f'SSIM: {ssim_val:.4f}')

In [ ]:
import matplotlib.pyplot as plt

lr = load_frame(EVAL_LR_PATH, EVAL_FRAME)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes,
                           [lr, sr, hr],
                           ['LR Input', f'SR (proposed2)\nPSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}', 'HR Ground Truth']):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{INFER_OUTPUT_DIR}/comparison_Subject0021.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure saved.')

---
## 6. Batch Evaluation — All Phonemes

Runs inference and computes metrics across all 11 test phonemes and saves a metrics CSV.

In [ ]:
import os, json
import nibabel as nib
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
import torch
import torch.nn.functional as F

PHONEMES      = ['ee', 'ff', 'mc', 'mm', 'mo', 'nn', 'oh', 'oo', 'sh', 'th', 'vv']
LR_TEMPLATE   = f'{DRIVE_ROOT}/SpeechSR_data/Synth_LR/LR_kspace_Subject0021_{{p}}.nii'
HR_TEMPLATE   = f'{DRIVE_ROOT}/SpeechSR_data/HR/HR_kspace_Subject0021_{{p}}.nii'
SR_OUTPUT_TPL = f'{INFER_OUTPUT_DIR}/proposed2_Subject0021_{{p}}.nii'

temporal_flag = '--temporal-window 3' if USE_TEMPORAL else '--temporal-window 1'
rows = []

for p in PHONEMES:
    lr_path = LR_TEMPLATE.format(p=p)
    hr_path = HR_TEMPLATE.format(p=p)
    sr_path = SR_OUTPUT_TPL.format(p=p)

    if not os.path.exists(lr_path):
        print(f'  Skipping {p} — LR file not found')
        continue

    os.system(f'python infer.py --model proposed2 --checkpoint "{INFER_CHECKPOINT}" '
              f'--input "{lr_path}" --output "{sr_path}" {temporal_flag}')

    if not os.path.exists(hr_path) or not os.path.exists(sr_path):
        continue

    sr = load_frame(sr_path)
    hr = load_frame(hr_path)
    if sr.shape != hr.shape:
        sr_t = torch.from_numpy(sr).unsqueeze(0).unsqueeze(0)
        sr = F.interpolate(sr_t, size=hr.shape, mode='bilinear', align_corners=False).squeeze().numpy()

    rows.append({'phoneme': p, 'psnr': psnr(hr, sr, data_range=1.0), 'ssim': ssim(hr, sr, data_range=1.0)})
    print(f'  {p}: PSNR={rows[-1]["psnr"]:.2f} dB  SSIM={rows[-1]["ssim"]:.4f}')

if rows:
    df = pd.DataFrame(rows)
    csv_path = f'{INFER_OUTPUT_DIR}/metrics_proposed2_Subject0021.csv'
    df.to_csv(csv_path, index=False)
    print(f'\nMean PSNR: {df["psnr"].mean():.2f} dB  |  Mean SSIM: {df["ssim"].mean():.4f}')
    print(f'Saved to: {csv_path}')

---
## 7. (Optional) Hyperparameter Optimisation

Uses Optuna to tune the loss weights and architecture hyperparameters for the standard (non-GAN) pipeline.  
Run this before GAN training to find good pixel-loss weights.

In [ ]:
HPO_OUTPUT_DIR = f'{OUTPUT_DIR}/hpo'
HPO_TRIALS     = 30
HPO_EPOCHS     = 25   # short runs per trial

!python train.py \
    --model               proposed2 \
    --input-dir           "{INPUT_DIR}" \
    --target-dir          "{TARGET_DIR}" \
    --output-dir          "{HPO_OUTPUT_DIR}" \
    --subjects            {SUBJECTS} \
    --val-subject         {VAL_SUBJECT} \
    --test-subject        {TEST_SUBJECT} \
    --patch-size          {PATCH_SIZE} \
    --augment \
    --perceptual-loss \
    --use-optuna \
    --n-trials            {HPO_TRIALS} \
    --hpo-epochs          {HPO_EPOCHS} \
    --train-after-hpo \
    --batch-size          {BATCH_SIZE} \
    --num-workers         {NUM_WORKERS}